# Technical Indicators and Binary Models in Python
### Companion material for Class 4 — *From Financial Data to Predictive Models*

**Algorithms and Data Analysis** · Professor Maru Hernández

---

This notebook brings together two things we usually see separately: **technical indicators**, which describe what already happened in the market, and the **models** that try to estimate something that has not happened yet.

One single idea connects all the material:

> A technical indicator is not a conclusion. It is an input variable for a model.

**Structure**

| Part | What we cover |
|---|---|
| **0** | Setup and market data |
| **I** | The five technical indicators of the course |
| **II** | Two binary models: credit risk and market direction |
| **III** | A third model with three options: buy, wait or sell |
| **IV** | The four models compared |

**How to use it.** Run the cells in order; each one uses what the previous one built. You do not need to write code: read what each block does and look at the results. If the data download fails, the notebook uses backup data and everything still works.

---
## Part 0 · Setup

We load the libraries and set the session parameters. There are only four things you can change: the stock, the start and end dates, and what share of the data we use for training.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

pd.set_option("display.width", 120)

# ------------------------------ PARAMETERS -----------------------------------
TICKER = "AAPL"          # change the ticker to work with another stock
START  = "2020-01-01"
END    = "2025-01-01"
TRAINING = 0.80          # oldest 80 % trains, most recent 20 % tests
# -----------------------------------------------------------------------------

print("Ready. We will work with:", TICKER)

Ready. We will work with: AAPL


### The market data

We download the price history. Two practical details worth knowing:

- The download **can fail** (no connection, misspelled ticker, provider rate limit). That is why there is backup data: the class never stops because of a network problem.
- The first thing we do with any price series is **sort it by date**. If the order is wrong, every indicator comes out wrong and no error warns you.

In [3]:
def simulated_prices(start, end):
    """Backup data if the download fails. NOT real market data."""
    dates = pd.bdate_range(start, end)
    random = np.random.default_rng(7)
    changes = random.normal(0.0003, 0.014, len(dates))
    close = 100 * np.exp(np.cumsum(changes))
    return pd.DataFrame({"Close": close}, index=dates)


try:
    import yfinance as yf
    data = yf.download(TICKER, start=START, end=END, progress=False)
    if isinstance(data.columns, pd.MultiIndex):       # sometimes it comes in two levels
        data.columns = data.columns.get_level_values(0)
    if data.empty:
        raise ValueError("the download came back empty")
    SOURCE = "Yahoo Finance"
except Exception as error:
    print("Could not download:", error)
    data = simulated_prices(START, END)
    SOURCE = "simulated data"

data = data.sort_index()            # chronological order always comes first
price = data["Close"]

print("Source:", SOURCE)
print("Observations:", len(data), "|", data.index.min().date(), "to", data.index.max().date())
data.head(3).round(2)

Source: Yahoo Finance
Observations: 1258 | 2020-01-02 to 2024-12-31


Price,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,72.27,72.33,71.03,71.28,135480400
2020-01-03,71.57,72.33,71.35,71.50,146322800
2020-01-06,72.14,72.18,70.44,70.69,118387200


---
# Part I · The five technical indicators

Each indicator answers a different financial question. You do not need to memorize formulas: you need to know which question each one answers.

| Indicator | Question it answers | Family |
|---|---|---|
| **Moving average (SMA)** | is the price above or below its recent average? | trend |
| **Volatility** | how much is the asset moving? | risk |
| **Bollinger Bands** | how far is the price from its normal range? | risk |
| **MACD** | is the short trend separating from the long one? | trend |
| **RSI** | was recent strength in gains or in losses? | momentum |

**About the empty first days.** A 200-day moving average does not exist before day 200, so the first rows have no value. That is normal and it is not an error. What matters is not filling them with zeros: a zero is a number, and the model would read it as real information.

In [ ]:
# --- The five indicators, one per line ---------------------------------------

ind = pd.DataFrame(index=data.index)
ind["Price"] = price
ind["Return"] = price.pct_change()                          # daily % change

# 1. Moving averages: the average of the last N days
ind["SMA_50"]  = price.rolling(50).mean()
ind["SMA_200"] = price.rolling(200).mean()

# 2. Volatility: how much the asset moves, in annual terms
ind["Volatility"] = ind["Return"].rolling(20).std() * np.sqrt(252)

# 3. Bollinger Bands: the "normal" range of the price
mean_20 = price.rolling(20).mean()
std_20  = price.rolling(20).std()
ind["Upper_band"] = mean_20 + 2 * std_20
ind["Lower_band"] = mean_20 - 2 * std_20

# 4. MACD: distance between a short trend and a long one
ind["MACD"] = price.ewm(span=12).mean() - price.ewm(span=26).mean()
ind["MACD_signal"] = ind["MACD"].ewm(span=9).mean()

# 5. RSI: strength of gains against losses, from 0 to 100
change = price.diff()
gains  = change.clip(lower=0).ewm(alpha=1/14).mean()
losses = (-change.clip(upper=0)).ewm(alpha=1/14).mean()
ind["RSI"] = 100 - 100 / (1 + gains / losses)

print(ind.tail(3).round(2))
print()
print("Days with no value at the start (warm-up):")
print(ind.isna().sum())

### The three indicators in one chart

**What to observe.** The three panels tell the same story in different vocabularies. When the price touches the upper band, the RSI is usually near 70 and the MACD is usually above its signal.

That overlap matters: if the indicators say almost the same thing, adding more of them to a model adds no information. It is an idea we will confirm later on.

In [ ]:
recent = ind.tail(180)           # roughly the last six months

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

# Panel 1: price, moving averages and bands
axes[0].plot(recent.index, recent["Price"], label="Price")
axes[0].plot(recent.index, recent["SMA_50"], label="SMA 50")
axes[0].plot(recent.index, recent["SMA_200"], label="SMA 200")
axes[0].fill_between(recent.index, recent["Lower_band"], recent["Upper_band"],
                     alpha=0.12, label="Bollinger Bands")
axes[0].set_ylabel("Price")
axes[0].set_title(TICKER + " with technical indicators")

# Panel 2: RSI with its reference levels
axes[1].plot(recent.index, recent["RSI"], color="purple", label="RSI 14")
axes[1].axhline(70, linestyle="--", linewidth=0.8, color="gray")
axes[1].axhline(30, linestyle="--", linewidth=0.8, color="gray")
axes[1].set_ylabel("RSI")

# Panel 3: MACD and its signal line
axes[2].plot(recent.index, recent["MACD"], label="MACD")
axes[2].plot(recent.index, recent["MACD_signal"], label="Signal")
axes[2].axhline(0, linewidth=0.8, color="gray")
axes[2].set_ylabel("MACD")

for axis in axes:
    axis.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

---
# Part II · Binary models

A **binary model** answers a yes-or-no question. Finance is full of them: will this client default? will the stock go up or down? is this transaction fraud?

We are going to solve two of them with the same model — a **logistic regression** — so we can compare them.

## II.A · Credit risk: will this loan default?

Each row is a credit applicant with their financial characteristics, and we want to estimate the probability that they will not pay:

$$\text{Default} = \begin{cases} 1 & \text{if the loan defaulted} \\ 0 & \text{if it was paid}\end{cases}$$

**Why we start here and not with the market.** Because in credit the signal **does exist**: a person's credit quality is informative and it persists over time. This exercise works as a control. When the market model later barely beats chance, we will know the problem is not the method but the phenomenon.

**About the data.** We use a sample loan book with the typical structure of a credit bureau: grade, term, debt-to-income ratio, income and number of open accounts.

In [ ]:
def simulated_credit(n=1200):
    """Sample loan book. Used if you do not have the real file."""
    random = np.random.default_rng(1)

    grade          = random.integers(0, 7, n)      # 0 = A (best) ... 6 = G (worst)
    long_term      = random.choice([0, 1], n, p=[0.72, 0.28])   # 1 = 60 months
    debt_to_income = np.round(np.abs(random.normal(18, 8, n)), 1)
    income         = np.round(np.exp(random.normal(11.0, 0.45, n)))
    accounts       = random.integers(2, 25, n)

    # True borrower risk: worse grade, more debt and less income
    risk = (-4.1 + 0.46*grade + 0.055*debt_to_income + 0.85*long_term
            - 1.15*np.log(income/50000) + 0.03*accounts)
    probability = 1 / (1 + np.exp(-risk))
    defaulted = (random.random(n) < probability).astype(int)

    return pd.DataFrame({
        "grade": grade,
        "long_term": long_term,
        "debt_to_income": debt_to_income,
        "log_income": np.log(income),
        "open_accounts": accounts,
        "Default": defaulted,
    })


credit = simulated_credit()

print("Loans in the dataset:", len(credit))
print("Default rate:", round(credit["Default"].mean(), 3))
print()
print(credit.head(3))

counts = credit["Default"].value_counts().sort_index()
plt.figure(figsize=(5, 3))
plt.bar(["Paid (0)", "Defaulted (1)"], counts.values, color=["#2a78d6", "#d03b3b"])
plt.ylabel("Loans")
plt.title("Most borrowers pay; default is the minority")
plt.tight_layout()
plt.show()

### The model

Five variables, one for each economic dimension of risk: **credit quality**, **term**, **leverage**, **repayment capacity** and **exposure**.

One important decision in this section: **here we can split the data at random.** Each row is a different client, not a moment in time of the same client. There is no future that can leak into the past. In Part II.B, with market data, doing this would be a serious mistake.

**What to observe.** The sign of each coefficient. A positive sign means that variable increases the probability of default, and all of them should make financial sense: a worse grade, more debt and a longer term raise the risk; more income lowers it.

In [ ]:
VARIABLES = ["grade", "long_term", "debt_to_income",
             "log_income", "open_accounts"]

X = credit[VARIABLES]
y = credit["Default"]

# This data is NOT a time series: each row is a different loan.
# That is why we can split it at random here.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=1, stratify=y)

credit_model = LogisticRegression(max_iter=1000)
credit_model.fit(X_train, y_train)

coefficients = pd.Series(credit_model.coef_[0], index=VARIABLES).round(3)

print("Training:", len(X_train), "loans | Test:", len(X_test))
print()
print("Coefficients (positive = higher risk of default):")
print(coefficients)

### Evaluation

The **confusion matrix** crosses what the model said with what actually happened. Every metric comes from there:

$$\text{sensitivity} = \frac{TP}{TP + FN}$$

That is: of everyone who actually defaulted, how many did we manage to catch?

**What to observe.** Three numbers and the distance between them.

- **Accuracy** barely beats the **benchmark**. That is not unusual: if only 25 % default, saying "nobody defaults" is already right 75 % of the time and it is a useless model.
- The **AUC** is much higher. That is not contradictory: accuracy depends on the 0.50 threshold, while the AUC measures whether the model **ranks** applicants correctly, from least to most risky, regardless of where you cut.
- A bank does not need to classify well at 0.50: it needs to rank well and then choose the cutoff according to how much risk it wants to take. That is why credit risk reports AUC.

In [ ]:
def results(y_true, y_pred, probability, names=("Yes", "No")):
    """Confusion matrix and basic metrics."""
    matrix = confusion_matrix(y_true, y_pred, labels=[1, 0])
    TP, FN = matrix[0]        # true positives, false negatives
    FP, TN = matrix[1]        # false positives, true negatives

    print(pd.DataFrame(matrix,
        index=["Actual " + names[0], "Actual " + names[1]],
        columns=["Predicted " + names[0], "Predicted " + names[1]]))

    accuracy = (TP + TN) / matrix.sum()
    benchmark = max(y_true.mean(), 1 - y_true.mean())
    sensitivity = TP / (TP + FN)

    print()
    print("Accuracy   :", round(accuracy, 3))
    print("Benchmark  :", round(benchmark, 3), "(always predict the most common class)")
    print("Sensitivity:", round(sensitivity, 3), " -> TP / (TP + FN)")
    print("AUC        :", round(roc_auc_score(y_true, probability), 3))

    return {"accuracy": accuracy, "benchmark": benchmark,
            "sensitivity": sensitivity}


credit_prob = credit_model.predict_proba(X_test)[:, 1]
credit_pred = (credit_prob > 0.50).astype(int)

credit_res = results(y_test, credit_pred, credit_prob,
                     names=("default", "pay"))

### The threshold is a business decision

There is nothing sacred about 0.50; it is a convention. Lowering it makes the model more suspicious: it catches more defaults, but it also rejects more clients who would have paid.

**What to observe.** How accuracy falls while sensitivity rises as the threshold drops. A bank does not choose the threshold that maximizes accuracy: it chooses the one that balances two very different costs. Rejecting a good client costs a lost fee; approving a defaulter costs the entire loan.

In [ ]:
rows = []
for threshold in [0.10, 0.15, 0.20, 0.30, 0.50]:
    prediction = (credit_prob > threshold).astype(int)
    matrix = confusion_matrix(y_test, prediction, labels=[1, 0])
    TP, FN = matrix[0]
    FP, TN = matrix[1]
    rows.append([threshold,
                 round((TP + TN) / matrix.sum(), 3),    # accuracy
                 round(TP / (TP + FN), 3),              # sensitivity
                 FP])                                    # good clients rejected

table = pd.DataFrame(rows, columns=["threshold", "accuracy", "sensitivity",
                                    "good clients rejected"])
print(table.to_string(index=False))

---
## II.B · Market direction: will it go up or down tomorrow?

Same model, different problem. But before solving it properly, it is worth seeing **how it gets solved badly**, because that is the most common mistake and the hardest one to spot.

### The mistake: predicting something we already know

Suppose we define the signal like this: it equals 1 if **today's** short moving average is above **today's** long one. And then we train a model to predict that signal using the MACD, the RSI and volatility, **also from today**.

Let us see what accuracy it gives.

In [ ]:
# TODAY's signal: the short average is above the long one
market = ind.dropna().copy()
market["SMA_12"] = market["Price"].rolling(12).mean()
market["SMA_26"] = market["Price"].rolling(26).mean()
market = market.dropna()
market["signal_today"] = (market["SMA_12"] > market["SMA_26"]).astype(int)

# The mistake: using SAME-DAY variables to "predict" that signal
today_variables = ["Volatility", "MACD", "MACD_signal", "RSI"]
split = int(len(market) * TRAINING)

flawed_model = LogisticRegression(max_iter=1000)
flawed_model.fit(market[today_variables][:split], market["signal_today"][:split])

flawed_accuracy = accuracy_score(market["signal_today"][split:],
                                 flawed_model.predict(market[today_variables][split:]))

print("Accuracy of this model:", round(flawed_accuracy, 3))
print()
print("It looks excellent, but it predicts nothing: the signal and the")
print("variables are computed from the same day's data.")

### Why that result is worthless

The accuracy is extremely high and it means nothing. The MACD is, by construction, the distance between a short trend and a long one; the signal is the comparison between a short average and a long one. We are asking the model whether the short one is above the long one, after handing it the distance between the short one and the long one as a clue. It is asking what two plus two is after saying the answer is four.

There are two problems and it is worth naming them separately:

1. **There is nothing to predict.** The signal describes today's state, not a future event.
2. **The variables contain the answer.** This is called `data leakage`. In a test it looks like a spectacular result; with real money it looks like a loss.

This is exactly the Class 4 warning: **a historical pattern is not a prediction**, and a result that is too good is grounds for suspicion before celebration.

### The correct version

Two changes, one line each:

- The **target** looks forward: it equals 1 if tomorrow's price is higher than today's. It is the only variable allowed to see the future.
- The **variables** look backward: they are shifted one day with `shift(1)`, so they only contain information that already existed before the move we want to predict.

And the data split is **chronological**: the earlier years train, the most recent ones test. Never at random.

In [ ]:
# 1. The target looks FORWARD
market["Direction"] = (market["Price"].shift(-1) > market["Price"]).astype(int)

# 2. The variables look BACKWARD: shifted one day with shift(1)
market["SMA_distance"] = market["Price"] / market["SMA_50"] - 1

MARKET_VARIABLES = ["Return", "RSI", "Volatility", "SMA_distance"]
for column in MARKET_VARIABLES:
    market[column + "_yesterday"] = market[column].shift(1)

COLUMNS = [c + "_yesterday" for c in MARKET_VARIABLES]
market = market.dropna(subset=COLUMNS + ["Direction"])

Xm = market[COLUMNS]
ym = market["Direction"]

# 3. CHRONOLOGICAL split: the past trains, the future evaluates
split = int(len(Xm) * TRAINING)
Xm_train, Xm_test = Xm[:split], Xm[split:]
ym_train, ym_test = ym[:split], ym[split:]

print("Trains:", Xm_train.index.min().date(), "to", Xm_train.index.max().date())
print("Tests :", Xm_test.index.min().date(), "to", Xm_test.index.max().date())
print("The dates do not overlap:", Xm_train.index.max() < Xm_test.index.min())
print()
print("Relationship of each variable with the target (should be close to zero):")
print(Xm.join(ym).corr()["Direction"].drop("Direction").round(3))

### Results of the correct model

**What to observe.** Four things, in this order:

1. **The range of the probability.** Almost every day sits right at 0.50. The model can barely tell one day from another, and that is evidence for the Efficient Market Hypothesis obtained from our own data.
2. **Accuracy against the benchmark.** If the model lands at 53 % and the benchmark is 52 %, the edge is one point and it could be pure luck of the period.
3. **The AUC.** If it falls below 0.50, the model ranked the days worse than chance in that period. It does not mean flipping the signal would work: it means the relationship it learned in the past stopped holding. That is the number one reason strategies that look good in a test fail afterwards.
4. **The simple rule as a benchmark.** If the trained model does not beat the moving average crossover rule, then the whole machine learning apparatus added nothing over what an analyst was already doing with a two-line rule.

Compare this result with Part II.A. Same model, same metric, incomparable results. The difference is not in the technique but in how much information each phenomenon actually contains.

In [ ]:
market_model = LogisticRegression(max_iter=1000)
market_model.fit(Xm_train, ym_train)

market_prob = market_model.predict_proba(Xm_test)[:, 1]
market_pred = (market_prob > 0.50).astype(int)

print("Probability of a rise: from", round(market_prob.min(), 3),
      "to", round(market_prob.max(), 3))
print()

if market_pred.min() == market_pred.max():
    print("At the 0.50 threshold the model always says the same thing.")
    print("We use the half of days with the highest probability so we can evaluate it.")
    market_pred = (market_prob > np.median(market_prob)).astype(int)
    print()

market_res = results(ym_test, market_pred, market_prob,
                     names=("up", "down"))

# Comparison with the plain technical rule, no model involved
simple_rule = (market["SMA_12"] > market["SMA_26"]).astype(int)[split:]
print()
print("Moving average crossover rule, no model:",
      round(accuracy_score(ym_test, simple_rule), 3))

---
# Part III · Three options instead of two

So far the answers have been yes or no. But many financial decisions have three options: **buy**, **wait** or **sell**.

For that we use **Linear Discriminant Analysis (LDA)**. It is another way of classifying observations, and it is worth knowing for three reasons:

1. When the groups are well separated, logistic regression becomes unstable; LDA does not.
2. With few observations, LDA tends to be more stable.
3. It naturally handles **more than two categories**, which is exactly what we need here.

The signal is built by combining two indicators we already computed:

$$\text{signal} = \begin{cases} +1 & \text{if the price breaks the upper band and the MACD confirms} \\ -1 & \text{if the price breaks the lower band and the MACD confirms} \\ 0 & \text{otherwise}\end{cases}$$

And with the same correction as the previous part: we predict **tomorrow's** signal using **today's** information.

**Watch out for the imbalance.** The "wait" class will dominate the sample, because a confirmed breakout is a rare event. A high accuracy here may simply mean the model learned to always say "wait". That is why we look at the hits **per class** and not just the total.

In [ ]:
# Three-state signal: buy (1), sell (-1) or wait (0)
buy  = (market["Price"] > market["Upper_band"]) & (market["MACD"] > market["MACD_signal"])
sell = (market["Price"] < market["Lower_band"]) & (market["MACD"] < market["MACD_signal"])

market["Signal"] = np.where(buy, 1, np.where(sell, -1, 0))
market["Signal_tomorrow"] = market["Signal"].shift(-1)     # we predict tomorrow's

three = market.dropna(subset=COLUMNS + ["Signal_tomorrow"])
Xl = three[COLUMNS]
yl = three["Signal_tomorrow"].astype(int)

print("How many days of each signal:")
print(yl.value_counts().sort_index())

split_l = int(len(Xl) * TRAINING)

lda_model = LinearDiscriminantAnalysis()
lda_model.fit(Xl[:split_l], yl[:split_l])
lda_pred = lda_model.predict(Xl[split_l:])

actual = yl[split_l:]
classes = [-1, 0, 1]
matrix = confusion_matrix(actual, lda_pred, labels=classes)

print()
print(pd.DataFrame(matrix,
      index=["Actual " + str(c) for c in classes],
      columns=["Predicted " + str(c) for c in classes]))

lda_accuracy = accuracy_score(actual, lda_pred)
print()
print("LDA accuracy:", round(lda_accuracy, 3))
print("Benchmark   :", round(actual.value_counts(normalize=True).max(), 3))
print()
for i, group in enumerate(classes):
    total = matrix[i].sum()
    if total > 0:
        print("Correct in class", group, ":", round(matrix[i, i] / total, 3))

if len(set(lda_pred)) == 1:
    print()
    print("The model ALWAYS predicted the same class: wait.")
    print("That is why its accuracy equals the benchmark: it adds no information.")
    print("This happens when one class dominates and the rest are rare events.")

---
# Part IV · The four models, side by side

In [ ]:
comparison = pd.DataFrame([
    ["Credit: default or pay",       "Logistic", "Loans (one row = one client)",
     "Random", round(credit_res["accuracy"], 3), round(credit_res["benchmark"], 3)],
    ["Market: same-day signal",      "Logistic", "Time series",
     "Chronological", round(flawed_accuracy, 3), np.nan],
    ["Market: up or down tomorrow",  "Logistic", "Time series",
     "Chronological", round(market_res["accuracy"], 3), round(market_res["benchmark"], 3)],
    ["Market: buy, wait or sell",    "LDA", "Time series",
     "Chronological", round(lda_accuracy, 3),
     round(actual.value_counts(normalize=True).max(), 3)],
], columns=["Problem", "Model", "Type of data", "Split",
            "Accuracy", "Benchmark"])

print(comparison.to_string(index=False))
print()
print("The only high accuracy in the table belongs to the badly framed model.")

ind.to_csv("indicators.csv")
print()
print("Saved: indicators.csv")

### What this material leaves you with

**1. Technical indicators are input variables, not conclusions.** The indicator describes the state of the market; the model is what risks an estimate.

**2. The same model gives very different results depending on the problem.** Logistic regression works well on credit risk and barely beats chance on market direction. It is not the method's fault: a person's credit quality is informative and persistent, while tomorrow's price already incorporates almost all of today's information.

**3. How you split the data depends on the type of data.** At random when each row is an independent case; chronologically when there is a timeline. Confusing them produces results that look good and do not hold.

**4. The highest accuracy in the whole notebook belongs to the badly framed model.** In finance, a result that is too good is the first thing to audit, not the first thing to show off.

**5. The metric follows the cost of the error.** In credit, sensitivity matters, because a false negative costs the entire loan. In markets, what matters is what survives trading costs. Accuracy alone is almost never the right metric.

### To practice

- Change `TICKER` to another stock and run everything again. Do the results hold?
- In Part II.B, raise the threshold from 0.50 to 0.60. There are fewer signals: does anything improve?
- In Part III, merge classes −1 and +1 into one ("there is a signal") and run the model as a two-option problem. Does it detect the rare events better?

---

*Companion material for Class 4 · The Class 5 models — decision trees, random forest and neural networks — will use exactly these same variables, this same target and this same chronological split.*